In [1]:
import scanpy as sc
import infercnvpy as cnv
import pandas as pd
from scipy import sparse as sp
import sys

sys.path.append('../../1_figure_CL_proof_of_concept/code/')
import utils_00 as gf_utils
large_data_dir = gf_utils.large_data_dir


In [2]:
### read in raw data (direct cellranger output, before any gene filtering)

adata_dir = large_data_dir + 'MPN_WTA/cr_BC007_1_1_sample_filtered_feature_bc_matrix.h5'
adata = sc.read_10x_h5(adata_dir)

adata_dir = large_data_dir + 'MPN_WTA/cr_BC007_1_2_sample_filtered_feature_bc_matrix.h5'
adata2 = sc.read_10x_h5(adata_dir)

adata = adata.concatenate(adata2, batch_key='expt', index_unique='-')

sc.pp.calculate_qc_metrics(adata, inplace = True)

min_counts = 500
min_genes = 250

sc.pp.filter_cells(adata,min_counts=min_counts)
sc.pp.filter_cells(adata,min_genes=min_genes)

### normalize and log-transform the data

sc.pp.normalize_total(adata, inplace = True)
sc.pp.log1p(adata)

/tmp/ipykernel_718681/403113506.py:9: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = adata.concatenate(adata2, batch_key='expt', index_unique='-')


In [3]:
## get processed adata to assign cell type labels, umap coordinates, and genotypes to adata

processed_adata = sc.read_h5ad(large_data_dir + 'MPN_WTA/MPN_1_BC007_genotyped.h5ad')

### filter adata to match processed_adata
adata = adata[adata.obs_names.isin(processed_adata.obs_names)].copy()
adata.obs = adata.obs.loc[processed_adata.obs_names]
adata.obs['cell_type'] = processed_adata.obs['cell_type']
adata.obsm['X_umap'] = processed_adata.obsm['X_umap']
adata.obsm['genotypes'] = processed_adata.obsm['genotypes']
adata.obs['pheno_leiden'] = processed_adata.obs['pheno_leiden']
adata.uns = processed_adata.uns

### label clones in adata
adata.obs['clone'] = pd.read_csv('../../6_figure_MPN_AML_phylogeny/output/clone_assignments.csv',index_col=0)['clone']

In [4]:
chr_table = pd.read_csv('../data/hg38_gencode_v27.txt', sep='\t', header=None, on_bad_lines='skip', index_col=0)
chr_table.columns = ['chromosome', 'start','end']
adata.var = adata.var.merge(chr_table, left_index=True, right_index=True, how='left')

In [5]:
%%time
cnv.tl.infercnv(
    adata,
    reference_key="clone",
    reference_cat=["TP53_CALR_het"],
    window_size=100,
    step=10, calculate_gene_values = True
)

  0%|          | 0/6 [00:00<?, ?it/s]

CPU times: user 3.2 s, sys: 13.3 s, total: 16.5 s
Wall time: 10min 1s


In [6]:
sp.save_npz('../output/infercnv_gene_values.npz', sp.csr_matrix(adata.layers['gene_values_cnv']))